In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/predicting-heart-disease-baseline-lgbm/__results__.html
/kaggle/input/predicting-heart-disease-baseline-lgbm/submission.csv
/kaggle/input/predicting-heart-disease-baseline-lgbm/test_preds.csv
/kaggle/input/predicting-heart-disease-baseline-lgbm/__notebook__.ipynb
/kaggle/input/predicting-heart-disease-baseline-lgbm/__output__.json
/kaggle/input/predicting-heart-disease-baseline-lgbm/oof_preds_lgbm.csv
/kaggle/input/predicting-heart-disease-baseline-lgbm/custom.css
/kaggle/input/predicting-heart-disease-logistic-regression/test_pred_lr.csv
/kaggle/input/predicting-heart-disease-logistic-regression/__results__.html
/kaggle/input/predicting-heart-disease-logistic-regression/oof_pred_lr.csv
/kaggle/input/predicting-heart-disease-logistic-regression/submission.csv
/kaggle/input/predicting-heart-disease-logistic-regression/__notebook__.ipynb
/kaggle/input/predicting-heart-disease-logistic-regression/__output__.json
/kaggle/input/predicting-heart-disease-logistic-regression/cust

In [2]:
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')

In [3]:
train.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [4]:
target = train['Heart Disease']

In [5]:
oof_pred_lr = pd.read_csv('/kaggle/input/predicting-heart-disease-logistic-regression/oof_pred_lr.csv')

In [6]:
oof_pred_lr.head()

,oof_pred_lr
0,0.996787
1,0.006291
2,0.011770
3,0.080762
4,0.998608


In [7]:
oof_preds_xgb = pd.read_csv('/kaggle/input/predicting-heart-disease-baseline-simple/oof_preds_xgb.csv')

In [8]:
oof_preds_xgb.head()

,oof_preds_xgb
0,0.997527
1,0.009253
2,0.011418
3,0.042925
4,0.998857


In [9]:
oof_preds_xgb_fe = pd.read_csv('/kaggle/input/notebooks/kumarkatariya/predicting-heart-disease-baseline-simple-fe/oof_preds_xgb.csv')

In [10]:
oof_preds_xgb_fe.head()

,oof_preds_xgb
0,0.998271
1,0.007205
2,0.010369
3,0.046679
4,0.999105


In [11]:
oof_preds_lgbm = pd.read_csv('/kaggle/input/predicting-heart-disease-baseline-lgbm/oof_preds_lgbm.csv')
oof_preds_lgbm.head()

,oof_preds_lgbm
0,0.997610
1,0.007978
2,0.008200
3,0.041993
4,0.998785


In [12]:
oof_preds_lgbm_ratio = pd.read_csv('/kaggle/input/notebooks/kumarkatariya/predicting-heart-disease-baseline-lgbm-ratio-fe/oof_preds_lgbm.csv')
oof_preds_lgbm_ratio.head()

,oof_preds_lgbm
0,0.997773
1,0.010660
2,0.013786
3,0.041120
4,0.999091


In [13]:
oof_preds_lgbm_seed = pd.read_csv('/kaggle/input/notebooks/kumarkatariya/predicting-heart-disease-baseline-lgbm-seed-avg/oof_preds_lgbm.csv')
oof_preds_lgbm_seed.head()

,oof_preds_lgbm
0,0.997724
1,0.011096
2,0.014302
3,0.036659
4,0.998717


In [14]:
oof_preds_tabM = pd.read_csv('/kaggle/input/notebooks/kumarkatariya/predicting-heart-disease-tabm/oof_preds_tabM.csv')
oof_preds_tabM.head()

,oof_preds_tabM
0,0.999710
1,0.007259
2,0.009153
3,0.047480
4,1.003369


In [15]:
test_preds = pd.read_csv('/kaggle/input/predicting-heart-disease-baseline-lgbm/test_preds.csv')
test_preds.head()

,id,test_preds
0,630000,0.968616
1,630001,0.011378
2,630002,0.987684
3,630003,0.009404
4,630004,0.166614


In [16]:
target

0         Presence
1          Absence
2          Absence
3          Absence
4         Presence
            ...   
629995     Absence
629996     Absence
629997    Presence
629998    Presence
629999     Absence
Name: Heart Disease, Length: 630000, dtype: object

In [17]:
target = target.map({'Presence':1,'Absence':0})

In [18]:
target

0         1
1         0
2         0
3         0
4         1
         ..
629995    0
629996    0
629997    1
629998    1
629999    0
Name: Heart Disease, Length: 630000, dtype: int64

In [19]:
train_df = pd.DataFrame({
    "pred_lr":oof_pred_lr['oof_pred_lr'],
    "pred_xgb":oof_preds_xgb['oof_preds_xgb'],
    "pred_lgbm":oof_preds_lgbm['oof_preds_lgbm'],
    "pred_tabM":oof_preds_tabM['oof_preds_tabM'],
    "pred_lgbm_seed":oof_preds_lgbm_seed['oof_preds_lgbm'],
    "pred_xgb_fe":oof_preds_xgb_fe['oof_preds_xgb'],
    "pred_lgbm_ratio":oof_preds_lgbm_ratio['oof_preds_lgbm'],
    "target":target
})

In [20]:
train_df.head()

,pred_lr,pred_xgb,pred_lgbm,pred_tabM,pred_lgbm_seed,pred_xgb_fe,pred_lgbm_ratio,target
0,0.996787,0.997527,0.997610,0.999710,0.997724,0.998271,0.997773,1
1,0.006291,0.009253,0.007978,0.007259,0.011096,0.007205,0.010660,0
2,0.011770,0.011418,0.008200,0.009153,0.014302,0.010369,0.013786,0
3,0.080762,0.042925,0.041993,0.047480,0.036659,0.046679,0.041120,0
4,0.998608,0.998857,0.998785,1.003369,0.998717,0.999105,0.999091,1


In [21]:
# test oofs

test_pred_lr = pd.read_csv('/kaggle/input/predicting-heart-disease-logistic-regression/test_pred_lr.csv')
test_pred_xgb = pd.read_csv('/kaggle/input/predicting-heart-disease-baseline-simple/test_preds.csv')
test_pred_tabM = pd.read_csv('/kaggle/input/notebooks/kumarkatariya/predicting-heart-disease-tabm/test_preds.csv')
test_pred_lgbm_seed= pd.read_csv('/kaggle/input/notebooks/kumarkatariya/predicting-heart-disease-baseline-lgbm-seed-avg/submission.csv')
test_pred_xgb_fe = pd.read_csv('/kaggle/input/notebooks/kumarkatariya/predicting-heart-disease-baseline-simple-fe/test_preds.csv')
test_pred_lgbm_ratio = pd.read_csv('/kaggle/input/notebooks/kumarkatariya/predicting-heart-disease-baseline-lgbm-ratio-fe/test_preds.csv')

In [22]:
test_pred_lgbm_seed.head()

,id,Heart Disease
0,630000,0.961336
1,630001,0.009702
2,630002,0.986638
3,630003,0.008956
4,630004,0.167901


In [23]:
test_pred_lgbm_ratio.head()

,id,test_preds
0,630000,0.947897
1,630001,0.012831
2,630002,0.986632
3,630003,0.011316
4,630004,0.188800


In [24]:
test_pred_xgb.head()

,id,test_preds
0,630000,0.962699
1,630001,0.010248
2,630002,0.987262
3,630003,0.007216
4,630004,0.173199


In [25]:
test_pred_lr.head()

,id,test_pred_lr
0,630000,0.966748
1,630001,0.003056
2,630002,0.994034
3,630003,0.009058
4,630004,0.115813


In [26]:
test_pred_tabM.head()

,id,test_preds
0,630000,0.950594
1,630001,0.006436
2,630002,0.992625
3,630003,-0.003741
4,630004,0.193763


In [27]:
test_ids = test_pred_lr['id']

In [28]:
test_df = pd.DataFrame({
    'pred_lr':test_pred_lr['test_pred_lr'],
    'pred_xgb':test_pred_xgb['test_preds'],
    'pred_lgbm':test_preds['test_preds'],
    'pred_tabM':test_pred_tabM['test_preds'],
    'pred_lgbm_seed':test_pred_lgbm_seed['Heart Disease'],
    'pred_xgb_fe':test_pred_xgb_fe['test_preds'],
    'pred_lgbm_ratio':test_pred_lgbm_ratio['test_preds']
})

In [29]:
test_df.head()

,pred_lr,pred_xgb,pred_lgbm,pred_tabM,pred_lgbm_seed,pred_xgb_fe,pred_lgbm_ratio
0,0.966748,0.962699,0.968616,0.950594,0.961336,0.978425,0.947897
1,0.003056,0.010248,0.011378,0.006436,0.009702,0.008614,0.012831
2,0.994034,0.987262,0.987684,0.992625,0.986638,0.989413,0.986632
3,0.009058,0.007216,0.009404,-0.003741,0.008956,0.006190,0.011316
4,0.115813,0.173199,0.166614,0.193763,0.167901,0.159524,0.188800


In [30]:
test_df.shape

(270000, 7)

In [31]:
#applying ridge as meta model

In [32]:
X = train_df.iloc[:,:7]

In [33]:
y = train_df['target']

In [34]:
X.head()

,pred_lr,pred_xgb,pred_lgbm,pred_tabM,pred_lgbm_seed,pred_xgb_fe,pred_lgbm_ratio
0,0.996787,0.997527,0.997610,0.999710,0.997724,0.998271,0.997773
1,0.006291,0.009253,0.007978,0.007259,0.011096,0.007205,0.010660
2,0.011770,0.011418,0.008200,0.009153,0.014302,0.010369,0.013786
3,0.080762,0.042925,0.041993,0.047480,0.036659,0.046679,0.041120
4,0.998608,0.998857,0.998785,1.003369,0.998717,0.999105,0.999091


In [35]:
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

In [36]:
from scipy.special import expit

In [37]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits = N_SPLITS, shuffle= True, random_state=42)

oof_pred_meta = np.zeros(len(X))
pred_test_meta = np.zeros(len(test_df))

for train_idx,val_idx in skf.split(X,y):
    X_train,X_val = X.iloc[train_idx].copy(),X.iloc[val_idx].copy()
    y_train,y_val = y.iloc[train_idx].copy(),y.iloc[val_idx].copy()

    model = RidgeClassifier(alpha=1.0)
    model.fit(X_train,y_train)

    val_pred = expit(model.decision_function(X_val))
    oof_pred_meta[val_idx] = val_pred

    fold_cv = roc_auc_score(y_val,val_pred)
    print(f'fold_cv:{fold_cv}') 
    
    pred_test_meta += expit(model.decision_function(test_df)) / N_SPLITS

final_cv = roc_auc_score(y,oof_pred_meta)
print(f'final_cv: {final_cv}') 

fold_cv:0.9558329068243621
fold_cv:0.9546882727083866
fold_cv:0.9555700009145673
fold_cv:0.9551147434364999
fold_cv:0.9559666745436705
final_cv: 0.955427022422926


In [38]:
sample = pd.read_csv('/kaggle/input/playground-series-s6e2/sample_submission.csv')

In [39]:
sample.head()

,id,Heart Disease
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0


In [40]:
submission = pd.DataFrame({
    "id":test_ids,
    "Heart Disease":pred_test_meta
})

In [41]:
submission.to_csv('submission.csv',index=False)